In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from lib.preprocess import preprocess
from lib.config import load_experiment_configs
from lib.experiments import run_experiments
from lib.models import make_pipeline
from lib.metrics import evaluate_regression

plt.rcParams["figure.figsize"] = (6, 6)
plt.rcParams["figure.dpi"] = 120

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
df = pd.read_csv("dsat.csv", dtype={352: "str", 353: "str"})
(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
    X_train_full,
    y_train_full,
    features,
) = preprocess(df, feature_set_num=1)

print("Train shape:", X_train.shape)
print("Val shape:  ", X_val.shape)
print("Test shape: ", X_test.shape)
print("Number of features after checking:", len(features))

In [ ]:
sota_configs = load_experiment_configs("configs/sota.yaml")

print("Number of sota experiments:", len(sota_configs))
for cfg in sota_configs:
    print(f"- {cfg.name} ({cfg.model_type})")

In [ ]:
sota_results_df, sota_pipelines = run_experiments(
    sota_configs,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
)

sota_results_df

In [ ]:
best_row = sota_results_df.iloc[0]
best_name = best_row["name"]
best_model_type = best_row["model_type"]

print("Best model by validation RMSE:")
print(best_row)

best_pipe_val = sota_pipelines[best_name]
best_name, best_model_type

In [ ]:
best_cfg = [c for c in sota_configs if c.name == best_name][0]

best_pipe_full = make_pipeline(best_cfg)
best_pipe_full.fit(X_train_full, y_train_full)

y_test_pred_full = best_pipe_full.predict(X_test)
final_metrics = evaluate_regression(y_test, y_test_pred_full)

print("Final test metrics (model trained on train_full):")
final_metrics

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_test_pred_full, s=5, alpha=0.3)

y_min = min(y_test.min(), y_test_pred_full.min())
y_max = max(y_test.max(), y_test_pred_full.max())
plt.plot([y_min, y_max], [y_min, y_max], "--")

plt.xlabel("True target")
plt.ylabel("Predicted target")
plt.title(f"Best model: {best_name} ({best_model_type})")
plt.grid(True)
plt.show()

In [ ]:
if best_model_type == "catboost":
    cb_model = best_pipe_full.named_steps["model"]
    importances = cb_model.get_feature_importance()
    
    fi = (
        pd.DataFrame({"feature": features, "importance": importances})
        .sort_values("importance", ascending=False)
    )
    
    fi.head(20)